In [1]:
%cd /kaggle/working
!git clone https://github.com/shujaabbasn/ATML.git
%cd /kaggle/working/ATML/pa4
!python -m pip install -q -r requirements.txt
!nvidia-smi

/kaggle/working
Cloning into 'ATML'...
remote: Enumerating objects: 899, done.
remote: Counting objects: 100% (347/347), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 899 (delta 133), reused 324 (delta 114), pack-reused 552 (from 1)
Receiving objects: 100% (899/899), 25.36 MiB | 28.35 MiB/s, done.
Resolving deltas: 100% (363/363), done.
/kaggle/working/ATML/pa4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.

In [2]:
# staff data repo (scripts + bundled files), then merge anything attached from earlier sessions
!git clone -q https://github.com/AbDu11aHHH/atml-pa4-datasets.git data_repo
!python kaggle_runs/attach_inputs.py
# skips files that already exist, so with data_prepare attached this is only a check (C4 streaming is slow otherwise)
!cd data_repo && python scripts/setup_data.py --public && python scripts/check_setup.py

nothing attached, data will be built from scratch
no revision lock yet
Locked upstream revisions in data/local_source_revisions.json
tokenizer_config.json: 7.23kB [00:00, 33.7MB/s]
vocab.json: 2.78MB [00:00, 21.8MB/s]
merges.txt: 1.67MB [00:00, 140MB/s]
tokenizer.json: 7.03MB [00:00, 202MB/s]
README.md: 7.02kB [00:00, 36.0MB/s]
data/train-00000-of-00001.parquet: 100%|███| 24.4M/24.4M [00:00<00:00, 39.1MB/s]
data/test-00000-of-00001.parquet: 100%|████| 6.11M/6.11M [00:00<00:00, 25.4MB/s]
data/validation-00000-of-00001.parquet: 100%|█| 6.32M/6.32M [00:00<00:00, 27.8MB
Generating test split: 100%|███| 10003/10003 [00:00<00:00, 302627.15 examples/s]
Generating validation split: 100%|█| 10042/10042 [00:00<00:00, 298929.74 example
Wrote data/public_locked/hellaswag_val_locked.jsonl.gz (755,571 bytes)
README.md: 7.93kB [00:00, 27.9MB/s]
main/train-00000-of-00001.parquet: 100%|███| 2.31M/2.31M [00:00<00:00, 8.61MB/s]
main/test-00000-of-00001.parquet: 100%|██████| 419k/419k [00:00<00:00, 2.71MB

In [3]:
# staff --prefix-banks through a shim: the staff script passes generator= to model.generate, which transformers 4.57.1 rejects
!python kaggle_runs/staff_prefix_banks.py && cd data_repo && python scripts/check_setup.py

tokenizer_config.json: 7.30kB [00:00, 29.2MB/s]
vocab.json: 2.78MB [00:00, 46.0MB/s]
merges.txt: 1.67MB [00:00, 99.8MB/s]
tokenizer.json: 7.03MB [00:00, 218MB/s]
tokenizer_config.json: 7.30kB [00:00, 26.8MB/s]
vocab.json: 2.78MB [00:00, 100MB/s]
merges.txt: 1.67MB [00:00, 160MB/s]
tokenizer.json: 7.03MB [00:00, 216MB/s]
Loading teacher
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 4.69MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
I0000 00:00:1795382723.908310     158 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
model.safetensors: 100%|████████████████████| 3.09G/3.09G [00:06<00:00, 473MB/s]
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 2.60MB/s]
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
teacher 20 / 120

In [4]:
!cd data_repo && python scripts/validate_assets.py && python -m unittest discover -s tests -v

PASS MQAR: 9216 deterministic rows, 9 cells
PASS KV: 180 prompt cases, char spans (not tokenizer lengths)
PASS short: 300 60
PASS hellaswag_val_ids 2000
PASS gsm8k_train_ids 500
PASS gsm8k_test_ids 60

PASS: available local assets structurally valid.
NOT CHECKED: dataset revision hashes, locked token windows, prefix banks, model retrieval accuracy.
test_mqar_determinism (test_datasets.DataTests.test_mqar_determinism) ... ok
test_retrieval_markers (test_datasets.DataTests.test_retrieval_markers) ... ok
test_short_test_disjoint (test_datasets.DataTests.test_short_test_disjoint) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.021s

OK


In [5]:
!zip -r /kaggle/working/data_repo.zip data_repo -x 'data_repo/.git/*'

  adding: data_repo/ (stored 0%)
  adding: data_repo/README.md (deflated 51%)
  adding: data_repo/tests/ (stored 0%)
  adding: data_repo/tests/__pycache__/ (stored 0%)
  adding: data_repo/tests/__pycache__/test_datasets.cpython-313.pyc (deflated 42%)
  adding: data_repo/tests/test_datasets.py (deflated 52%)
  adding: data_repo/data/ (stored 0%)
  adding: data_repo/data/prefix_banks/ (stored 0%)
  adding: data_repo/data/prefix_banks/prefix_banks_locked.jsonl.gz (deflated 0%)
  adding: data_repo/data/public_locked/ (stored 0%)
  adding: data_repo/data/public_locked/gsm8k_train_locked.jsonl.gz (deflated 0%)
  adding: data_repo/data/public_locked/gsm8k_test_locked.jsonl.gz (stored 0%)
  adding: data_repo/data/public_locked/wikitext2_test_windows_locked.jsonl.gz (deflated 0%)
  adding: data_repo/data/public_locked/c4_development_locked.jsonl.gz (stored 0%)
  adding: data_repo/data/public_locked/c4_calibration_locked.jsonl.gz (deflated 0%)
  adding: data_repo/data/public_locked/hellaswag_val